# KBO deep_learning_state — Colab runner

Thin wrapper. No training logic lives in this notebook; every cell shells out to
`src/deep_learning_state/`. Edit the code on the Mac, push, re-run here.

**Runtime > Change runtime type > GPU** before running anything.

## 1. Clone / pull

In [ ]:
REPO = 'https://github.com/Gromiit/kbo-control.git'
import os, pathlib
if pathlib.Path('/content/kbo/.git').exists():
    !cd /content/kbo && git pull --ff-only
else:
    !git clone $REPO /content/kbo
os.chdir('/content/kbo')
!git rev-parse --short HEAD


## 2. Dependencies + CUDA check

In [ ]:
!bash scripts/setup_colab.sh

## 3. Attach the shards

Shards are built on the Mac and are NOT in git. They arrive as one tarball,
`kbo_seq_L32.tgz` (198 MB compressed, 2.2 GB extracted).

**Extract to /content, do not symlink or mmap out of Drive.** The dataset
memory-maps `.npy`; mmap over the Drive FUSE mount is a network round trip per
page and turns a 40-second epoch into tens of minutes. /content has ~70 GB.

Checkpoints and results go the other way -- straight to Drive via
`KBO_CKPT` / `KBO_EXP`, so a disconnect costs nothing.

Override the archive location with `KBO_DATA_ARCHIVE` if it is not at the
default `MyDrive/kbo/kbo_seq_L32.tgz`.


In [ ]:
import os
from google.colab import drive; drive.mount('/content/drive')

os.environ.setdefault('KBO_DATA_ARCHIVE',
                      '/content/drive/MyDrive/kbo/kbo_seq_L32.tgz')
# optional: sums written next to the Mac build, uploaded alongside the tarball
os.environ.setdefault('KBO_DATA_SHA256',
                      '/content/drive/MyDrive/kbo/SHA256SUMS_L32.txt')

# shards: Drive -> local disk. macOS tar ships `._*` sidecars; they are inert
# (the shard glob is anchored, so it never matches them) and cost ~30 KB.
!mkdir -p /content/kbo/data
!tar xzf "$KBO_DATA_ARCHIVE" -C /content/kbo/data
!du -sh /content/kbo/data/sequences && ls /content/kbo/data/sequences

# checkpoints + results stay on Drive
os.environ['KBO_CKPT'] = '/content/drive/MyDrive/kbo/out/checkpoints'
os.environ['KBO_EXP']  = '/content/drive/MyDrive/kbo/out'
!mkdir -p "$KBO_CKPT"


### 3b. Verify what arrived

Inventory, shapes, and the window invariants that are visible in the shards
alone (left-padding, `length`, p_v9 in valid only). If `SHA256SUMS_L32.txt`
was uploaded it also proves these are byte-identical to the tree the leakage
audit passed on the Mac.

`audit.py` itself is a **Mac** step -- it rebuilds every window from
`data/folds/*.parquet` to compare, and those 1.5 GB of folds are not uploaded.
Season isolation, row_id overlap, window-vs-parquet equality, scaler
provenance and p_v9-vs-OOF alignment were established there before upload.


In [ ]:
import os, pathlib
args = ['--seasons', '2023,2024', '--sequence-length', '32']
sums = os.environ.get('KBO_DATA_SHA256', '')
if pathlib.Path(sums).exists():
    args += ['--sha256', sums]
else:
    print(f'no checksum file at {sums!r} -- structural checks only')
ARGS = ' '.join(args)
!python -m src.deep_learning_state.check_shards $ARGS


## 4. Smoke test (always first)

In [ ]:
!SMOKE_ONLY=1 bash scripts/train_colab.sh

## 5. Full training

Only after the smoke test passes **and** you have been told to run it.
Seeds run sequentially — one GPU, one model at a time.

In [ ]:
!CONFIG=configs/gru_full.yaml SEEDS='42' bash scripts/train_colab.sh

## 6. Results

`KBO_EXP` already points at Drive, so `results.csv`, the per-epoch traces and
the checkpoints are written there directly. Nothing to copy.


In [ ]:
!ls -la "$KBO_EXP"
!cat "$KBO_EXP/results.csv"
